# Run-book — build the test AML images for one commit

**Proves:** both Azure ML images exist under the test names, built from one pinned rise-eo commit (#234).
**Needs:** an Azure login in `~/.config/rise-eo/config.toml` (with `image_registry`); the commit pushed to GitHub. Seconds if the images exist, else 10–20 minutes per image.

Run it cell by cell. Paste back the **Summary** output, or the failing cell's message.

## Before you start

- **Kernel:** `.venv` from the main checkout (`rise-eo/.venv/bin/python`). Setup checks it.
- **Resume:** after a restart, run Setup and Settings, then any step.
- **When done:** Restart & Clear All Outputs, then save (a test refuses saved outputs).

In [ ]:
# Setup -- nothing to change here.
import sys
from pathlib import Path

VENV = Path(sys.prefix)
assert VENV.name == ".venv" and (VENV.parent / "pyproject.toml").exists(), (
    f"Wrong kernel: {sys.prefix}. Select Kernel -> Python Environments -> the main checkout's .venv.")
MAIN_CHECKOUT = VENV.parent

# Import this notebook's own src/, not the main checkout's (matters in a worktree).
CHECKOUT = Path.cwd().parent
assert Path.cwd().name == "runbooks" and (CHECKOUT / "src" / "rise_eo").is_dir(), (
    f"The kernel started in {Path.cwd()}, not in this notebook's runbooks/ folder.")
if str(CHECKOUT / "src") not in sys.path:
    sys.path.insert(0, str(CHECKOUT / "src"))
import rise_eo

assert Path(rise_eo.__file__).is_relative_to(CHECKOUT / "src"), (
    f"rise-eo was already imported from {rise_eo.__file__}. Restart the kernel, then run this cell first.")
print("rise-eo:", rise_eo.__file__)

## Settings

In [ ]:
# Settings
import json
import re

import rise_eo
import rise_eo.aml
from rise_eo.image import ImageDefinition
from rise_eo.image import registry as image_registry

cfg = rise_eo.config.load()      # resource group, workspace, image registry: private, read at run time

# The full 40-character sha of the commit to build: `git rev-parse origin/main` for the latest main.
COMMIT = "<40-character sha>"
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT), "set COMMIT to a full 40-character sha"

# Test names: never the formal rise-eo-aml-env / rise-eo-infer-sklearn, and no alias (#234).
BASE_NAME = "rise-eo-aml-env-test"
INFER_NAME = "rise-eo-infer-sklearn-test"
RISE_EO_REF = f"git+https://github.com/nasaharvest/rise-eo@{COMMIT}"   # a git ref, so pip records the commit

OUT = MAIN_CHECKOUT / "tests" / "outputs" / "test_images" / COMMIT[:12]   # gitignored
OUT.mkdir(parents=True, exist_ok=True)

## Step 1 — ensure both images

Registers both images from `RISE_EO_REF` and reuses any that already exist. **PASS:** both report `reused`. After a new build the cell stops and prints a Studio link: open it, wait for **Build status: Succeeded**, then re-run the cell. **Time:** 10–20 minutes per new build.

In [ ]:
# Step 1 -- ensure both environments; a new build stops here until Studio says Succeeded.
BASE = ImageDefinition(name=BASE_NAME, rise_eo=RISE_EO_REF, extras=("local", "azure", "mpc"))
INFER = BASE.derive(name=INFER_NAME, extra_pip=("scikit-learn", "joblib"))

envs = {}
for key, defn in (("aml_env", BASE), ("infer_env", INFER)):
    env = rise_eo.aml.ensure_environment(
        defn, registry=cfg.image_registry,
        resource_group=cfg.resource_group, workspace=cfg.workspace, storage="azure",
    )
    print(key, env.ref, "reused" if env.reused else "JUST BUILT")
    assert env.reused, (
        f"{env.ref} was just built: open {env.build_url} , wait for `Build status: Succeeded`, "
        "then re-run this cell.")
    envs[key] = {"ref": env.ref, "registry_ref": env.registry_ref}
(OUT / "envs.json").write_text(json.dumps(envs, indent=2))
print("written:", OUT / "envs.json")

## Step 2 — check what was registered

Reads both images' recorded definitions back. **PASS:** both are the test names and install rise-eo from exactly `RISE_EO_REF`.

In [ ]:
# Step 2 -- read envs.json and the registry's recorded definitions back, then assert.
envs = json.loads((OUT / "envs.json").read_text())
assert envs["aml_env"]["ref"].startswith(f"{BASE_NAME}:"), envs["aml_env"]["ref"]
assert envs["infer_env"]["ref"].startswith(f"{INFER_NAME}:"), envs["infer_env"]["ref"]
for key, env in envs.items():
    resolved = image_registry.resolve(env["registry_ref"], cfg.image_registry)
    assert resolved.definition.get("rise_eo") == RISE_EO_REF, (
        f"{key}: the registry says rise_eo={resolved.definition.get('rise_eo')!r}, expected {RISE_EO_REF!r}")
print("PASS: both images are test images built from", COMMIT)

## Summary — paste this back

Put `AZ_ENV` into the next run-book's Settings.

In [ ]:
import json

envs = json.loads((OUT / "envs.json").read_text())
summary = {
    "commit": COMMIT,
    "AZ_ENV": envs["aml_env"]["ref"],
    "inference_env": envs["infer_env"]["ref"],
}
print(json.dumps(summary, indent=2))